In [ ]:
import kagglehub
!pip install segmentation-models-pytorch

path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path deta files", path)

In [ ]:
def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask

In [ ]:
import os
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms
import torch
import numpy as np

class SUIMDataset(Dataset):
    def __init__(self, root_path, transform=None):
        self.transform = transform
        self.image_path = ""
        self.mask_path = ""

        for root, dirs, files in os.walk(root_path):
            if 'images' in dirs and 'masks' in dirs:
                self.image_path = os.path.join(root, 'images')
                self.mask_path = os.path.join(root, 'masks')
                break

        if not self.image_path:
            raise FileNotFoundError("راجع ملفاتك")

        self.image_files = sorted([f for f in os.listdir(self.image_path) if f.endswith(('.jpg', '.png'))])

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_full_path = os.path.join(self.image_path, img_name)
        mask_name = img_name.replace('.jpg', '.png')
        mask_full_path = os.path.join(self.mask_path, mask_name)

        image = Image.open(img_full_path).convert("RGB")
        mask = Image.open(mask_full_path).convert("L")

        if self.transform:
            image = self.transform(image)
            mask = transforms.Resize((256, 256), interpolation=transforms.InterpolationMode.NEAREST)(mask)

        mask = torch.from_numpy(np.array(mask)).long()
        mask = remap_mask(mask)
        return image, mask

transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

dataset = SUIMDataset(root_path=path, transform=transform)
train_loader = DataLoader(dataset, batch_size=8, shuffle=True)
print("uvv hgwmn", len(dataset))

In [ ]:
import segmentation_models_pytorch as smp

model = smp.Unet(
    encoder_name="efficientnet-b1",
    encoder_weights="imagenet",
    in_channels=3,
    classes=8,
)

In [ ]:
def train_fn(model, loader, optimizer, criterion, device):
    model.train()
    epoch_loss = 0
    for images, masks in loader:
        images, masks = images.to(device), masks.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
    return epoch_loss / len(loader)

def validate_fn(model, loader, criterion, device):
    model.eval()
    epoch_loss = 0
    with torch.no_grad():
        for images, masks in loader:
            images, masks = images.to(device), masks.to(device)
            outputs = model(images)
            loss = criterion(outputs, masks)
            epoch_loss += loss.item()
    return epoch_loss / len(loader)

In [ ]:
import matplotlib.pyplot as plt
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

train_losses = []
epochs = 5

for epoch in range(epochs):
    loss = train_fn(model, train_loader, optimizer, criterion, device)
    train_losses.append(loss)
    print(f"Epoch {epoch+1}, Loss: {loss:.4f}")

plt.plot(train_losses)
plt.title("Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
model.eval()
images, masks = next(iter(train_loader))
images, masks = images.to(device), masks.to(device)

with torch.no_grad():
    preds = model(images)
    preds = torch.softmax(preds, dim=1)
    preds = torch.argmax(preds, dim=1)

plt.figure(figsize=(15, 10))
for i in range(3):
    plt.subplot(3, 3, i*3 + 1); plt.imshow(images[i].cpu().permute(1, 2, 0)); plt.title("Original Image")
    plt.subplot(3, 3, i*3 + 2); plt.imshow(masks[i].cpu()); plt.title("Ground Truth")
    plt.subplot(3, 3, i*3 + 3); plt.imshow(preds[i].cpu()); plt.title("Prediction")
plt.tight_layout()
plt.show()